In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/js042710/second3t1k/CLEARDATA/clean_31.1.2026.listens.csv
/kaggle/input/datasets/js042710/second3t1k/CLEARDATA/clean_24.1.2026.listens.csv
/kaggle/input/datasets/js042710/second3t1k/CLEARDATA/clean_9.1.2026.listens.csv
/kaggle/input/datasets/js042710/second3t1k/CLEARDATA/clean_17.1.2026.listens.csv
/kaggle/input/datasets/js042710/second3t1k/CLEARDATA/clean_10.1.2026.listens.csv
/kaggle/input/datasets/js042710/second3t1k/CLEARDATA/clean_8.1.2026.listens.csv
/kaggle/input/datasets/js042710/second3t1k/CLEARDATA/clean_15.1.2026.listens.csv
/kaggle/input/datasets/js042710/second3t1k/CLEARDATA/clean_14.1.2026.listens.csv
/kaggle/input/datasets/js042710/second3t1k/CLEARDATA/clean_30.1.2026.listens.csv
/kaggle/input/datasets/js042710/second3t1k/CLEARDATA/clean_26.1.2026.listens.csv
/kaggle/input/datasets/js042710/second3t1k/CLEARDATA/clean_16.1.2026.listens.csv
/kaggle/input/datasets/js042710/second3t1k/CLEARDATA/clean_1.2.2026.listens.csv
/kaggle/input/datasets/js042710

Load dữ liệu và định hình lại thời gian

In [2]:
import os
import glob
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')

# 1. Cấu hình đường dẫn (Trỏ thẳng vào thư mục CLEARDATA của dataset)
CONFIG = {
    'data_dir': '/kaggle/input/datasets/js042710/second3t1k/CLEARDATA'
}

print("1. ĐANG ĐỌC VÀ GỘP DỮ LIỆU...")
# Lấy danh sách file và sắp xếp
all_files = sorted(glob.glob(os.path.join(CONFIG['data_dir'], '*.csv')))

df_list = []
# Chỉ định các cột cần thiết để tối ưu RAM tối đa
cols_to_use = ['user_id', 'timestamp', 'recording_msid', 'artist_name']

for f in all_files:
    try:
        # Đọc từng file với danh sách cột đã chỉ định
        df_temp = pd.read_csv(f, usecols=cols_to_use)
        df_list.append(df_temp)
    except Exception as e:
        print(f"Lỗi đọc file {os.path.basename(f)}: {e}")

# Gộp thành 1 DataFrame duy nhất
df_full = pd.concat(df_list, ignore_index=True)
del df_list # Xóa list trung gian để giải phóng RAM

# Ép kiểu thời gian ngay lập tức
df_full['timestamp'] = pd.to_datetime(df_full['timestamp'])

print(f"-> Gộp thành công! Tổng số dòng logs: {df_full.shape[0]:,}")

1. ĐANG ĐỌC VÀ GỘP DỮ LIỆU...
-> Gộp thành công! Tổng số dòng logs: 80,631,061


In [3]:
print("\n2. ĐANG TRÍCH XUẤT ĐẶC TRƯNG NGƯỜI DÙNG...")

# Xác định giờ ban đêm (Giả định: 23h đêm đến 4h sáng)
df_full['hour'] = df_full['timestamp'].dt.hour
df_full['is_night'] = df_full['hour'].apply(lambda x: 1 if (x >= 23 or x <= 4) else 0)

# Gom nhóm dữ liệu theo từng User
user_profile = df_full.groupby('user_id').agg(
    total_listens=('recording_msid', 'count'),  # Tổng lượt nghe
    night_listens=('is_night', 'sum'),          # Lượt nghe ban đêm
    first_listen=('timestamp', 'min'),          # Lần nghe đầu tiên
    last_listen=('timestamp', 'max')            # Lần nghe cuối cùng
)

# Tính tỷ lệ cú đêm (Night Ratio)
user_profile['night_ratio'] = user_profile['night_listens'] / user_profile['total_listens']

# Tính số ngày hoạt động (Active Days)
user_profile['active_days'] = (user_profile['last_listen'] - user_profile['first_listen']).dt.days + 1

# Tính độ trung thành với nghệ sĩ (Loyalty Score)
top_artist = df_full.groupby(['user_id', 'artist_name']).size().groupby('user_id').max()
user_profile['loyalty_score'] = top_artist / user_profile['total_listens']

# Lọc bộ feature cuối cùng và xử lý NaN
features = ['total_listens', 'night_ratio', 'active_days', 'loyalty_score']
X = user_profile[features].fillna(0)

print(f"-> Đã tạo hồ sơ hoàn chỉnh cho {user_profile.shape[0]:,} người dùng.")


2. ĐANG TRÍCH XUẤT ĐẶC TRƯNG NGƯỜI DÙNG...
-> Đã tạo hồ sơ hoàn chỉnh cho 23,376 người dùng.


In [4]:
print("\n3. ĐANG CHẠY K-MEANS VÀ GÁN NHÃN...")

# Chuẩn hóa dữ liệu về cùng thang đo
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Chạy thuật toán K-Means với 3 nhóm
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
user_profile['cluster'] = kmeans.fit_predict(X_scaled)

# Lấy giá trị tâm cụm để phân tích
cluster_centers = user_profile.groupby('cluster')[features].mean()
display(cluster_centers.round(3))

# Logic tự động gọi tên các nhóm
labels = {}
for cluster_id, row in cluster_centers.iterrows():
    # Nhóm cày đêm nhiều nhất
    if row['night_ratio'] == cluster_centers['night_ratio'].max():
        labels[cluster_id] = "Cú đêm"
    # Nhóm nghe ít nhất (hoặc số ngày hoạt động ít nhất)
    elif row['total_listens'] == cluster_centers['total_listens'].min():
        labels[cluster_id] = "Người mới"
    # Nhóm nghe nhiều và độ trung thành cao
    else:
        labels[cluster_id] = "Fan cứng"

# Map kết quả vào bảng dữ liệu
user_profile['User_Type'] = user_profile['cluster'].map(labels)

print("\n--- PHÂN BỐ SỐ LƯỢNG NGƯỜI DÙNG THEO NHÓM ---")
print(user_profile['User_Type'].value_counts())


3. ĐANG CHẠY K-MEANS VÀ GÁN NHÃN...


,total_listens,night_ratio,active_days,loyalty_score
cluster,,,,
0,1159.065,0.190,50.044,0.140
1,181.136,0.220,11.274,0.769
2,104611.944,0.184,4994.013,0.042



--- PHÂN BỐ SỐ LƯỢNG NGƯỜI DÙNG THEO NHÓM ---
User_Type
Fan cứng    19832
Cú đêm       3544
Name: count, dtype: int64
